In [1]:
%pip install plotly
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# =========================================================
# 1. Load & filter
# =========================================================
df = pd.read_csv("Fresh Entry.csv")

df = df[df["employmentTypes"].isin(["Full Time", "Permanent"])]
df = df[df["positionLevels"] == "Fresh/entry level"]
df = df[df["metadata_newPostingDate"] >= 45047]          # 2023-05-01
df = df.drop_duplicates(subset="metadata_jobPostId")

# =========================================================
# 2. Clean salary (min/max -> midpoint, drop placeholders)
# =========================================================
for c in ["salary_minimum", "salary_maximum"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[
    (df["salary_type"] == "Monthly")
    & df["salary_minimum"].notna()
    & df["salary_maximum"].notna()
].copy()

# swap if min > max
swap = df["salary_minimum"] > df["salary_maximum"]
df.loc[swap, ["salary_minimum", "salary_maximum"]] = (
    df.loc[swap, ["salary_maximum", "salary_minimum"]].values
)

df["salary_mid"] = (df["salary_minimum"] + df["salary_maximum"]) / 2

# drop obvious placeholder salaries (e.g. 1, 10, 12) for FT fresh-entry monthly
df = df[df["salary_mid"] >= 800]

# =========================================================
# 3. Applications & repost
# =========================================================
df["applications"]  = pd.to_numeric(df["metadata_totalNumberJobApplication"],
                                     errors="coerce").fillna(0)
df["repost_count"]  = pd.to_numeric(df["metadata_repostCount"],
                                     errors="coerce").fillna(0)

# =========================================================
# 4. Explode categories (fractional weight, avoid double-count)
# =========================================================
df["categories_list"] = df["categories"].apply(json.loads)
df["n_cat"]           = df["categories_list"].apply(len)

ex = df.explode("categories_list").copy()
ex["sector"] = ex["categories_list"].apply(lambda x: x["category"])
ex["weight"] = 1.0 / ex["n_cat"]

# winsorize salary per sector (1st–99th pct) to tame remaining outliers
def _wins(s):
    if len(s) < 5:
        return s
    return s.clip(s.quantile(0.01), s.quantile(0.99))

ex["salary_mid_w"] = ex.groupby("sector")["salary_mid"].transform(_wins)

# =========================================================
# 5. Sector-level aggregation
# =========================================================
agg = ex.groupby("sector").agg(
    postings  = ("weight",     "sum"),     # weighted count
    salary_y  = ("salary_mid_w","median"),
    apps_x    = ("applications", "median"),
    repost    = ("repost_count", "median"),
).reset_index()

# keep top-N sectors by posting volume for readability
TOP_N = 12
top = agg.sort_values("postings", ascending=False).head(TOP_N).copy()

# quadrant thresholds = medians of displayed sectors
x_mid = top["apps_x"].median()
y_mid = top["salary_y"].median()

# plot bounds
xmin, xmax = 0, max(top["apps_x"].max() * 1.15, 1)
ymin = top["salary_y"].min() * 0.85
ymax = top["salary_y"].max() * 1.15

# =========================================================
# 6. Build figure
# =========================================================
fig = go.Figure()

# --- quadrant shading (below points) ---
fig.add_shape(type="rect", x0=xmin, x1=x_mid, y0=y_mid, y1=ymax,
              fillcolor="rgba(244,63,94,0.10)",  line_width=0, layer="below")   # talent gap
fig.add_shape(type="rect", x0=x_mid, x1=xmax, y0=y_mid, y1=ymax,
              fillcolor="rgba(59,130,246,0.10)", line_width=0, layer="below")   # competitive
fig.add_shape(type="rect", x0=x_mid, x1=xmax, y0=ymin, y1=y_mid,
              fillcolor="rgba(245,158,11,0.10)", line_width=0, layer="below")   # high competition
fig.add_shape(type="rect", x0=xmin, x1=x_mid, y0=ymin, y1=y_mid,
              fillcolor="rgba(107,114,128,0.10)", line_width=0, layer="below")  # undesirable

# --- quadrant divider lines ---
fig.add_shape(type="line", x0=x_mid, x1=x_mid, y0=ymin, y1=ymax,
              line=dict(dash="dash", color="gray", width=1))
fig.add_shape(type="line", x0=xmin, x1=xmax, y0=y_mid, y1=y_mid,
              line=dict(dash="dash", color="gray", width=1))

# --- bubbles ---
size = top["postings"] / top["postings"].max() * 60 + 15

fig.add_trace(go.Scatter(
    x=top["apps_x"],
    y=top["salary_y"],
    mode="markers+text",
    text=top["sector"],
    textposition="top center",
    textfont=dict(size=10),
    marker=dict(
        size=size,
        color=top["repost"],
        colorscale="Reds",
        showscale=True,
        colorbar=dict(title="Median<br>repost count"),
        line=dict(color="white", width=1),
        opacity=0.85,
    ),
    customdata=np.stack([top["postings"], top["repost"]], axis=-1),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Median applications: %{x:.0f}<br>"
        "Median salary mid: S$%{y:,.0f}/mo<br>"
        "Weighted postings: %{customdata[0]:.1f}<br>"
        "Median repost: %{customdata[1]:.0f}<extra></extra>"
    ),
    name="",
))

# --- quadrant labels ---
annotations = [
    dict(x=xmin, y=ymax, xanchor="left",  yanchor="top",
         text="🎯 Talent gap<br>high pay · low applications · reposts",
         showarrow=False, align="left", font=dict(size=11, color="#be123c")),
    dict(x=xmax, y=ymax, xanchor="right", yanchor="top",
         text="🔥 Competitive & attractive<br>high pay · high applications",
         showarrow=False, align="right", font=dict(size=11, color="#1d4ed8")),
    dict(x=xmax, y=ymin, xanchor="right", yanchor="bottom",
         text="📣 High competition<br>low pay · high applications (volume hiring)",
         showarrow=False, align="right", font=dict(size=11, color="#b45309")),
    dict(x=xmin, y=ymin, xanchor="left",  yanchor="bottom",
         text="⚠️ Undesirable / poorly advertised<br>low pay · low applications",
         showarrow=False, align="left", font=dict(size=11, color="#4b5563")),
]

fig.update_layout(
    title="TalentGap Finder — where do fresh-entry sectors sit?",
    xaxis_title="Median applications per posting",
    yaxis_title="Median fresh-entry salary midpoint (S$/month)",
    template="plotly_white",
    height=650, width=1050,
    annotations=annotations,
    margin=dict(l=80, r=40, t=80, b=60),
)

fig.show()
# fig.write_html("talent_gap_bubble.html")
# fig.write_image("talent_gap_bubble.png", scale=2)   # needs kaleido

  Using cached narwhals-2.28.0-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 475.5 kB/s  0:00:20 eta 0:00:01
Using cached narwhals-2.28.0-py3-none-any.whl (490 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]
Note: you may need to restart the kernel to use updated packages.
